In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import numpy as np

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [2]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()
print(f"Loaded model: {model_name}")
print(f"Hidden size: {model.config.hidden_size}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: distilbert-base-uncased
Hidden size: 768


In [3]:
dataset = load_dataset("glue", "mrpc", split="validation")
dataset = dataset.select(range(400))
print("Dataset split: glue/mrpc validation")
print("Subset: deterministic head slice of first 400 examples")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])

Dataset split: glue/mrpc validation
Subset: deterministic head slice of first 400 examples
Number of examples: 400
Example row:
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}


In [4]:
def encode_texts_cls(texts, batch_size=64, max_length=128):
    all_embeddings = []
    for start_idx in range(0, len(texts), batch_size):
        batch_texts = texts[start_idx:start_idx + batch_size]
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(cls_embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)

sentence1_list = dataset["sentence1"]
sentence2_list = dataset["sentence2"]
labels = dataset["label"]

emb1 = encode_texts_cls(sentence1_list, batch_size=64, max_length=128)
emb2 = encode_texts_cls(sentence2_list, batch_size=64, max_length=128)

emb1_norms = torch.norm(emb1, dim=1)
emb2_norms = torch.norm(emb2, dim=1)

emb1_normalized = F.normalize(emb1, p=2, dim=1)
emb2_normalized = F.normalize(emb2, p=2, dim=1)
cosine_similarities = F.cosine_similarity(emb1_normalized, emb2_normalized).tolist()

threshold = 0.80
predictions = [1 if score >= threshold else 0 for score in cosine_similarities]
confidences = [abs(score - threshold) for score in cosine_similarities]

print(f"Completed embedding inference for {len(predictions)} examples.")
print("Sentence vector method: CLS token from DistilBERT last hidden state")
print(f"Fixed cosine similarity threshold: {threshold}")

Completed embedding inference for 400 examples.
Sentence vector method: CLS token from DistilBERT last hidden state
Fixed cosine similarity threshold: 0.8


In [5]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, predictions)

positive_scores = [score for score, label in zip(cosine_similarities, labels) if label == 1]
negative_scores = [score for score, label in zip(cosine_similarities, labels) if label == 0]
mean_positive_similarity = sum(positive_scores) / len(positive_scores)
mean_negative_similarity = sum(negative_scores) / len(negative_scores)

emb1_norms_np = emb1_norms.numpy()
emb2_norms_np = emb2_norms.numpy()
all_norms_np = np.concatenate([emb1_norms_np, emb2_norms_np])

print("Evaluation metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)
print(f"Mean cosine similarity | label=1: {mean_positive_similarity:.4f}")
print(f"Mean cosine similarity | label=0: {mean_negative_similarity:.4f}")
print("Embedding norm summaries:")
print(f"sentence1 norms | mean={emb1_norms_np.mean():.4f} std={emb1_norms_np.std():.4f} min={emb1_norms_np.min():.4f} max={emb1_norms_np.max():.4f}")
print(f"sentence2 norms | mean={emb2_norms_np.mean():.4f} std={emb2_norms_np.std():.4f} min={emb2_norms_np.min():.4f} max={emb2_norms_np.max():.4f}")
print(f"all norms      | mean={all_norms_np.mean():.4f} std={all_norms_np.std():.4f} min={all_norms_np.min():.4f} max={all_norms_np.max():.4f}")

Evaluation metrics:
Accuracy : 0.6850
Precision: 0.6850
Recall   : 1.0000
F1       : 0.8131
Confusion matrix:
[[  0 126]
 [  0 274]]
Mean cosine similarity | label=1: 0.9745
Mean cosine similarity | label=0: 0.9571
Embedding norm summaries:
sentence1 norms | mean=12.2645 std=0.4836 min=10.8140 max=15.1448
sentence2 norms | mean=12.2710 std=0.5020 min=10.6176 max=15.0337
all norms      | mean=12.2677 std=0.4929 min=10.6176 max=15.1448


In [6]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}
num_examples_to_show = 10

mismatch_indices = [
    i for i, (true_label, pred_label) in enumerate(zip(labels, predictions))
    if true_label != pred_label
]

def disagreement_score(i):
    score = cosine_similarities[i]
    true_label = labels[i]
    if true_label == 1:
        return threshold - score
    return score - threshold

ranked_mismatches = sorted(mismatch_indices, key=disagreement_score, reverse=True)

print(f"Total mismatches: {len(mismatch_indices)}")

for rank, i in enumerate(ranked_mismatches[:num_examples_to_show], start=1):
    row = dataset[i]
    true_label = labels[i]
    pred_label = predictions[i]
    score = cosine_similarities[i]
    disagreement = disagreement_score(i)
    print(f"Hardest mismatch {rank}")
    print(f"index: {i}")
    print(f"sentence1: {row['sentence1']}")
    print(f"sentence2: {row['sentence2']}")
    print(f"true label: {true_label} ({label_map[true_label]})")
    print(f"pred label: {pred_label} ({label_map[pred_label]})")
    print(f"cosine similarity: {score:.4f}")
    print(f"score disagreement from expected side of threshold: {disagreement:.4f}")
    print("-" * 80)

Total mismatches: 126
Hardest mismatch 1
index: 104
sentence1: " I don 't know if the person I 'm talking to now may end up being someone else at another time that may not follow the rules , " Parrish said .
sentence2: " I don 't know whether the person I 'm talking to now may end up being someone else , " Parrish said .
true label: 0 (not_paraphrase)
pred label: 1 (paraphrase)
cosine similarity: 0.9966
score disagreement from expected side of threshold: 0.1966
--------------------------------------------------------------------------------
Hardest mismatch 2
index: 354
sentence1: It decided instead to issue them before the stock market opened Monday after the downgrade of its debt late Friday by Moody 's , the credit rating agency .
sentence2: It decided instead to issue them before the stock market opened Monday to counteract the downgrade of its debt late Friday by Moody 's to one step above junk status .
true label: 0 (not_paraphrase)
pred label: 1 (paraphrase)
cosine similarity: 0

In [7]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("embedding_method=cls_token_last_hidden_state")
print("inference_method=separate_sentence_embeddings_with_cosine_similarity")
print("dataset_split=glue/mrpc validation")
print("dataset_subset=first_400_examples")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"threshold={threshold}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"mean_positive_similarity={mean_positive_similarity:.4f}")
print(f"mean_negative_similarity={mean_negative_similarity:.4f}")
print(f"embedding_norm_mean={all_norms_np.mean():.4f}")
print(f"embedding_norm_std={all_norms_np.std():.4f}")
print(f"num_mismatches={len(mismatch_indices)}")

RESULT SUMMARY
model=distilbert-base-uncased
embedding_method=cls_token_last_hidden_state
inference_method=separate_sentence_embeddings_with_cosine_similarity
dataset_split=glue/mrpc validation
dataset_subset=first_400_examples
device=mps
num_examples=400
threshold=0.8
accuracy=0.6850
precision=0.6850
recall=1.0000
f1=0.8131
mean_positive_similarity=0.9745
mean_negative_similarity=0.9571
embedding_norm_mean=12.2677
embedding_norm_std=0.4929
num_mismatches=126
